In [1]:
# python
import sys
import os
import importlib
# columnar analysis
from coffea import processor
from coffea.nanoevents import NanoEventsFactory, NanoAODSchema
import awkward as ak
from dask.distributed import Client, performance_report
# local
sidm_path = str(os.getcwd()).split("/sidm")[0]
# sidm_path = str(sys.path[0]).split("/sidm")[0]
if sidm_path not in sys.path: sys.path.insert(1, sidm_path)
from sidm.tools import utilities, sidm_processor, scaleout, cutflow
from sidm.tools import llpnanoaodschema
# always reload local modules to pick up changes during development
importlib.reload(utilities)
importlib.reload(sidm_processor)
importlib.reload(scaleout)
# plotting
import matplotlib.pyplot as plt
utilities.set_plot_style()
%matplotlib inline
from tqdm.notebook import tqdm
import coffea.util
import numpy as np
import mplhep as hep
import yaml

In [2]:
client = scaleout.make_dask_client("tls://localhost:8786")
client

Connection method: Direct,
Dashboard: /user/dongyub.lee@cern.ch/proxy/8787/status,
Comm: tls://192.168.235.40:8786,Workers: 0
Dashboard: /user/dongyub.lee@cern.ch/proxy/8787/status,Total threads: 0
Started: 1 hour ago,Total memory: 0 B


In [3]:
runner = processor.Runner(
    # executor=processor.IterativeExecutor(),
    executor=processor.DaskExecutor(client=client),
    # schema=NanoAODSchema,
    schema = llpnanoaodschema.LLPNanoAODSchema,
    # maxchunks=1,
    # chunksize=100000,
    skipbadfiles=True,
)

In [4]:
channels = [
    "data_control_region_1muLj_cosmic_veto",
    "data_control_region_1egmLj_cosmic_veto",
    "data_control_region_1muLj_1egmLj_cosmic_veto",
    "data_control_region_2muLj_cosmic_veto",
    "data_control_region_1muLj_cosmic_veto_both_pass",
    "data_control_region_1muLj_cosmic_veto_both_fail",
    "data_control_region_1egmLj_cosmic_veto_both_pass",
    "data_control_region_1egmLj_cosmic_veto_both_fail",
    "data_control_region_1muLj_1egmLj_cosmic_veto_both_pass",
    "data_control_region_1muLj_1egmLj_cosmic_veto_both_fail",
    "data_control_region_2muLj_cosmic_veto_both_pass",
    "data_control_region_2muLj_cosmic_veto_both_fail",
],

p = sidm_processor.SidmProcessor(
    channels,
    ["mother_tracking_base"],
    unweighted_hist=False,
)

In [4]:
yaml_file_path = '../../configs/ntuples/signal_2mu2e_v10.yaml'
# Open and read the YAML file
with open(yaml_file_path, 'r') as file:
    data = yaml.safe_load(file)
signals_2mu2e_all = list(data["llpNanoAOD_v2"]["samples"].keys())

max_files_2mu2e = -1
fileset_2mu2e = utilities.make_fileset(signals_2mu2e_all, "llpNanoAOD_v2", max_files=max_files_2mu2e, location_cfg="signal_2mu2e_v10.yaml")
output_2mu2e = runner.run(fileset_2mu2e, treename="Events", processor_instance=p)
coffea.util.save(output_2mu2e, f"signal_2mu2e.coffea")

{'TTJets': 1, 'QCD_Pt800To1000': 1, 'DYJetsToMuMu_M50': 1}

In [ ]:
yaml_file_path = '../../configs/ntuples/signal_4mu_v10.yaml'
# Open and read the YAML file
with open(yaml_file_path, 'r') as file:
    data = yaml.safe_load(file)
signals_4mu_all = list(data["llpNanoAOD_v2"]["samples"].keys())

max_files_4mu = -1
fileset_4mu = utilities.make_fileset(signals_4mu_all, "llpNanoAOD_v2", max_files=max_files_4mu, location_cfg="signal_4mu_v10.yaml")
output_4mu = runner.run(fileset_4mu, treename="Events", processor_instance=p)
coffea.util.save(output_4mu, f"signal_4mu.coffea")

In [5]:
samples_bkg = [
    "TTJets",
    
    # "QCD_Pt15To20",
    "QCD_Pt20To30",
    "QCD_Pt30To50",
    "QCD_Pt50To80",
    "QCD_Pt80To120", 
    # "QCD_Pt120To170",
    "QCD_Pt170To300", 
    "QCD_Pt300To470",
    "QCD_Pt470To600", 
    "QCD_Pt600To800", 
    # "QCD_Pt800To1000",
    "QCD_Pt1000", 
    
    "DYJetsToMuMu_M10to50",
    "DYJetsToMuMu_M50",

    "WW",
    "WZ",
    "ZZ",
]

In [6]:
fileset = utilities.make_fileset(samples_bkg[0:60], 
                                 "skimmed_llpNanoAOD_v2", 
                                 location_cfg="backgrounds.yaml",
                                 max_files = -1,
                                )

In [7]:
out = {}
for i, bkg in enumerate(samples_bkg):
    
    print(f"Processing {bkg}")
    fileset_one_bkg = {samples_bkg[i]:fileset.get(samples_bkg[i])}
    
    output = runner.run(fileset_one_bkg, treename='Events', processor_instance=p)

    #Add this sample's output to the out variable
    out[bkg] = output["out"][bkg]

    #Save output to a file!!
    out_file_name = bkg + ".coffea"
    coffea.util.save(output,out_file_name)

Processing TTJets
[                                        ] | 0% Completed | 38.1s3.4s

Exception: Failed processing file: WorkItem(dataset='TTJets', filename='root://xcache//store/group/lpcmetx/SIDM/Backgrounds/2018_v2/Skims/TTJets_TuneCP5/skimmed_output2_182.root', treename='Events', entrystart=0, entrystop=12762, fileuuid=b']\xa4Z\xfc\x13[\x11\xf0\xb5\x11\xf9\xbe\xe1\x83\xbe\xef', usermeta={'skim_factor': 0.2746591272963762, 'year': '2018', 'is_data': False})